In [2]:
!pip install transformers seqeval

In [3]:
!wget https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-train.conllu
!wget https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-dev.conllu

--2026-04-04 04:23:55--  https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-train.conllu
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 15029817 (14M) [text/plain]
Saving to: ‘en_ewt-ud-train.conllu.1’

en_ewt-ud-train.con 100%[===================>]  14.33M  31.3MB/s    in 0.5s    

2026-04-04 04:23:56 (31.3 MB/s) - ‘en_ewt-ud-train.conllu.1’ saved [15029817/15029817]

--2026-04-04 04:23:56--  https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-dev.conllu
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP requ

In [4]:
def read_conllu(file):
    sentences = []
    labels = []

    tokens = []
    pos_tags = []

    with open(file, "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith("#") or line.strip() == "":
                if tokens:
                    sentences.append(tokens)
                    labels.append(pos_tags)
                    tokens = []
                    pos_tags = []
                continue

            parts = line.strip().split("\t")
            if "-" in parts[0] or "." in parts[0]:
                continue

            tokens.append(parts[1])
            pos_tags.append(parts[3])

    return sentences, labels

In [5]:
train_tokens, train_labels = read_conllu("en_ewt-ud-train.conllu")
val_tokens, val_labels = read_conllu("en_ewt-ud-dev.conllu")

In [6]:
label_list = list(set(tag for sent in train_labels for tag in sent))

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [14]:
def tokenize_and_align(tokens, labels):
    tokenized = tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        padding='max_length',
        max_length=128
    )

    word_ids = tokenized.word_ids()
    label_ids = []
    prev = None

    for word_idx in word_ids:
        if word_idx is None:
            label_ids.append(-100)
        elif word_idx != prev:
            label_ids.append(label2id[labels[word_idx]])
        else:
            label_ids.append(-100)
        prev = word_idx

    tokenized["labels"] = label_ids
    return tokenized

In [9]:
train_data = [tokenize_and_align(t, l) for t, l in zip(train_tokens, train_labels)]
val_data = [tokenize_and_align(t, l) for t, l in zip(val_tokens, val_labels)]

In [10]:
from datasets import Dataset

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

In [11]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized be

In [12]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

In [16]:
!pip install -U transformers

In [18]:
from transformers import TrainingArguments, Trainer

In [19]:
args = TrainingArguments(
    output_dir="out",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2
)

In [20]:
from seqeval.metrics import precision_score, recall_score, f1_score

def compute_metrics(p):
    preds, labels = p
    preds = preds.argmax(axis=2)

    true_preds = []
    true_labels = []

    for pred, lab in zip(preds, labels):
        temp_p = []
        temp_l = []
        for p_val, l_val in zip(pred, lab):
            if l_val != -100:
                temp_p.append(id2label[p_val])
                temp_l.append(id2label[l_val])
        true_preds.append(temp_p)
        true_labels.append(temp_l)

    return {
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds)
    }

In [21]:
train_dataset = train_dataset.select(range(300))
val_dataset = val_dataset.select(range(100))

In [22]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [23]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=76, training_loss=1.8409733019377057, metrics={'train_runtime': 825.951, 'train_samples_per_second': 0.726, 'train_steps_per_second': 0.092, 'total_flos': 39199828838400.0, 'train_loss': 1.8409733019377057, 'epoch': 2.0})

In [31]:
trainer.state.global_step = 0
trainer.state.epoch = 0
trainer.state.is_world_process_zero = True

In [28]:
sentence = "John works at Google in California"

tokens = sentence.split()
inputs = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)

outputs = model(**inputs)
preds = outputs.logits.argmax(dim=2)[0].tolist()

word_ids = inputs.word_ids()
final = []
prev = None

for i, word_idx in enumerate(word_ids):
    if word_idx is None or word_idx == prev:
        continue
    final.append((tokens[word_idx], id2label[preds[i]]))
    prev = word_idx

for word, tag in final:
    print(word, "→", tag)

John → PROPN
works → VERB
at → ADP
Google → PROPN
in → ADP
California → PROPN
